# Twin — Evaluation Notebook
Training curves, GraphGA candidates visualisation, DQN version comparison.

In [ ]:
# Cell 1 — Imports and setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')

try:
    from rdkit import Chem
    from rdkit.Chem import QED, Descriptors
    RDKIT_OK = True
except ImportError:
    print('RDKit not available — some cells will be skipped')
    RDKIT_OK = False

plt.rcParams['figure.dpi'] = 110
print('Setup OK')

In [ ]:
# Cell 2 — Load GraphGA candidates
import os

validated_path = '../graphga_validated_candidates.csv'
top_path = '../graphga_top_candidates.csv'

if os.path.exists(validated_path):
    df_val = pd.read_csv(validated_path)
    print(f'graphga_validated_candidates.csv  shape={df_val.shape}')
    print('Columns:', list(df_val.columns))
else:
    print('graphga_validated_candidates.csv not found — skipping')
    df_val = None

if os.path.exists(top_path):
    df_top = pd.read_csv(top_path)
    print(f'\ngraphga_top_candidates.csv  shape={df_top.shape}')
    print('Columns:', list(df_top.columns))
else:
    print('graphga_top_candidates.csv not found — skipping')
    df_top = None

if df_top is not None:
    sort_col = 'composite' if 'composite' in df_top.columns else df_top.columns[-1]
    print(f'\nTop 10 candidates (sorted by {sort_col}):')
    display(df_top.sort_values(sort_col, ascending=False).head(10))

In [ ]:
# Cell 3 — Scatter plot: QED vs MW, colour = composite score
df = df_top if df_top is not None else df_val

if df is not None and 'qed' in df.columns and 'mw' in df.columns:
    color_col = 'composite' if 'composite' in df.columns else None
    c = df[color_col] if color_col else 'steelblue'

    fig, ax = plt.subplots(figsize=(8, 5))
    sc = ax.scatter(df['mw'], df['qed'], c=c, cmap='RdYlGn', s=80, edgecolors='k', linewidths=0.4)

    if color_col:
        plt.colorbar(sc, ax=ax, label='Composite score (reward proxy)')

    # Lipinski limits
    ax.axvline(500, color='red', linestyle='--', linewidth=1, label='Lipinski MW ≤ 500')
    ax.axhline(0.5, color='orange', linestyle='--', linewidth=1, label='QED ≥ 0.5')

    ax.set_xlabel('Molecular Weight (Da)')
    ax.set_ylabel('QED')
    ax.set_title('GraphGA top candidates — drug-likeness profile')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Required columns (qed, mw) not found — skipping plot')

In [ ]:
# Cell 4 — ChEMBL pre-training results (from README)
epochs = list(range(1, 11))
train_rmse = [0.8412, 0.6231, 0.5184, 0.4502, 0.4038, 0.3714, 0.3482, 0.3305, 0.3178, 0.3076]
val_rmse   = [0.7923, 0.5841, 0.4892, 0.4253, 0.3821, 0.3521, 0.3302, 0.3142, 0.2088, 0.3201]

best_epoch = int(np.argmin(val_rmse)) + 1
best_val   = min(val_rmse)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs, train_rmse, 'o-', label='Train RMSE')
ax.plot(epochs, val_rmse,   's-', label='Val RMSE')
ax.annotate(
    f'Best val={best_val:.4f}\n(epoch {best_epoch})',
    xy=(best_epoch, best_val),
    xytext=(best_epoch + 0.3, best_val + 0.04),
    arrowprops=dict(arrowstyle='->', color='black'),
    fontsize=9
)
ax.set_xlabel('Epoch')
ax.set_ylabel('RMSE')
ax.set_title('ChEMBL GNN pre-training — train vs val RMSE')
ax.legend()
ax.set_xticks(epochs)
plt.tight_layout()
plt.show()

df_pretrain = pd.DataFrame({'epoch': epochs, 'train_rmse': train_rmse, 'val_rmse': val_rmse})
display(df_pretrain)

In [ ]:
# Cell 5 — QSAR CCLE training curve
qsar_epochs    = [1, 5, 10, 15, 20]
qsar_train     = [0.821, 0.612, 0.538, 0.503, 0.489]
qsar_val       = [0.794, 0.591, 0.521, 0.487, 0.472]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(qsar_epochs, qsar_train, 'o-', label='Train RMSE')
ax.plot(qsar_epochs, qsar_val,   's-', label='Val RMSE')
ax.annotate(
    'Val RMSE = 0.472\n≈ 0.71 log µM\n≈ 5-fold error in µM space',
    xy=(20, 0.472),
    xytext=(14, 0.56),
    arrowprops=dict(arrowstyle='->', color='black'),
    fontsize=8
)
ax.set_xlabel('Epoch')
ax.set_ylabel('RMSE (log µM)')
ax.set_title('QSAR CCLE training curve')
ax.legend()
ax.set_xticks(qsar_epochs)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 — DQN version history comparison
dqn_versions = [
    {'version': 'v1 (demo)', 'valid_pct': 45.0, 'best_reward': 0.31, 'best_smiles': 'C(C)(C)NCC(O)c1ccc(O)cc1'},
    {'version': 'v2',        'valid_pct': 62.0, 'best_reward': 0.48, 'best_smiles': 'CC(C)NCC(O)c1ccc(O)c(O)c1'},
    {'version': 'v3',        'valid_pct': 71.0, 'best_reward': 0.55, 'best_smiles': 'O=C1NC(=O)c2ccccc21'},
    {'version': 'v3.4',      'valid_pct': 78.0, 'best_reward': 0.62, 'best_smiles': 'c1ccc2c(c1)cc1ccc3cccc4ccc2c1c34'},
    {'version': 'v3.5',      'valid_pct': 81.0, 'best_reward': 0.67, 'best_smiles': 'c1ccc2[nH]ccc2c1'},
    {'version': 'v3.6',      'valid_pct': 83.0, 'best_reward': 0.71, 'best_smiles': 'c1ccc2c(c1)ccc1ccccc12'},
    {'version': 'v3.7',      'valid_pct': 85.0, 'best_reward': 0.74, 'best_smiles': 'c1ccc2c(c1)[nH]c1ccccc12'},
]

df_dqn = pd.DataFrame(dqn_versions)
display(df_dqn[['version', 'valid_pct', 'best_reward', 'best_smiles']])

fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.viridis(np.linspace(0.2, 0.85, len(df_dqn)))
bars = ax.barh(df_dqn['version'], df_dqn['best_reward'], color=colors)
ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=9)
ax.set_xlabel('Best reward')
ax.set_title('DQN version history — best reward per version')
ax.set_xlim(0, df_dqn['best_reward'].max() * 1.15)
plt.tight_layout()
plt.show()

## Cell 7 — Known Limitations

| # | Limitation | Scientific impact |
|---|-----------|-------------------|
| 1 | **Drug features = random vectors** (no PubChem SMILES mapping yet) | IC50 model learns from omics only, not molecular structure — drug branch is essentially a learnable embedding |
| 2 | **Mutations modality absent** (MAF parsing issue) | Only 2 / 3 omics modalities used; mutation signatures known to be predictive for some drug classes |
| 3 | **Validation split is random** (not leave-drug-out / leave-cell-out) | Generalisation to entirely unseen drugs or cell lines not proven |
| 4 | **No simple baseline** (Ridge regression, Random Forest) for comparison | Architectural benefit of the Bi-Int block over shallow models remains unquantified |
| 5 | **SA score missing from DQN reward** | Synthetically inaccessible molecules (high SA score) are not penalised; generated candidates may be difficult to synthesise |


In [ ]:
# Cell 8 — Baseline comparison
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

baseline_path = '../Dataset/baseline_results.csv'

if not os.path.exists(baseline_path):
    print(f'baseline_results.csv not found at {baseline_path}')
    print('Run `python baseline_models.py` first to generate it.')
    df_bl = None
else:
    df_bl = pd.read_csv(baseline_path)
    print(f'Loaded {len(df_bl)} rows from {baseline_path}')
    display(df_bl.to_string(index=False))

if df_bl is not None:
    # ── Bar plot: RMSE on random split ──────────────────────────────────────
    df_random = df_bl[df_bl['Split'] == 'Random'].copy()

    # Add Bi-Int reference row
    biint_row = pd.DataFrame([{
        'Model': 'Bi-Int (omics only)',
        'Split': 'Random',
        'RMSE':  0.472,
        'Pearson_r':  float('nan'),
        'Spearman_r': float('nan'),
    }])
    df_plot = pd.concat([df_random, biint_row], ignore_index=True)

    colors = ['#4C72B0', '#55A868', '#C44E52', '#DD8452']
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(df_plot['Model'], df_plot['RMSE'], color=colors[:len(df_plot)],
                  edgecolor='k', linewidth=0.6)
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
    ax.set_ylabel('RMSE (normalised log µM)')
    ax.set_title('IC50 prediction — Baseline vs Bi-Int (random split)')
    ax.set_ylim(0, df_plot['RMSE'].max() * 1.2)
    plt.xticks(rotation=15, ha='right')
    plt.tight_layout()
    plt.show()

    # ── Full results table across all splits ────────────────────────────────
    print('\nFull baseline results:')
    display(df_bl)

In [ ]:
# Cell 9 — Drug SMILES mapping summary
import os
import pandas as pd

smiles_path = '../Dataset/ccle_drug_smiles.csv'

if not os.path.exists(smiles_path):
    print(f'ccle_drug_smiles.csv not found at {smiles_path}')
    print('Run `python fetch_drug_smiles.py` first to generate it.')
else:
    df_smi = pd.read_csv(smiles_path)
    n_total  = len(df_smi)
    n_mapped = df_smi['smiles'].notna().sum()
    n_missed = n_total - n_mapped

    print(f'Drug SMILES mapping summary')
    print(f'  Total drugs : {n_total}')
    print(f'  Mapped      : {n_mapped} ({100 * n_mapped / n_total:.1f}%)')
    print(f'  Not found   : {n_missed} ({100 * n_missed / n_total:.1f}%)')

    # Source breakdown
    if 'source' in df_smi.columns:
        print(f"\nSource breakdown:")
        print(df_smi['source'].value_counts(dropna=False).to_string())

    # 5 examples
    mapped = df_smi[df_smi['smiles'].notna()]
    print(f"\n5 example mappings:")
    for _, row in mapped.head(5).iterrows():
        smi_preview = str(row['smiles'])[:60] + ('...' if len(str(row['smiles'])) > 60 else '')
        print(f"  {row['drug_name']:<35s} → {smi_preview}")

    # List unmapped drugs
    missed = df_smi[df_smi['smiles'].isna()]['drug_name'].tolist()
    if missed:
        print(f"\nUnmapped drugs ({len(missed)}):")
        for m in missed:
            print(f"  - {m}")